# **PHASE 04 — LABEL / TARGERT COLUMN CREATION**
### *Forward Return Threshold Labeling Strategy*
- Tranforming historical data into a supervised learning dataset for predicting profitable trades.

**How it works**
- “If I buy this stock today, will I make at least 5% profit within the next 10 days?”
- if ≥ 5% >>> 1 (BUY) else Not
- model learns which stocks today will gain ≥5% within 10 days

**How our Code Implements This**
- **Step 1** — Get Future Price >>> **LEAD(close, 10)** >>> This gets the closing price 10 records later >>>
- **Step 2** — Calculate Future Return >>> **future_return = (future_price / current_price ) - 1**
- **Step 3** — Apply Profit Threshold >>> PROFIT_THRESHOLD = 0.05 >>> if future_return = 10% -> 1

### Helpers

In [25]:
import os
from datetime import datetime
import numpy as np
import pandas as pd

# ================ CONFIG ===================

pd.set_option('display.max_columns', None)

# CSV Folder
CSV_FOLDER = "output_csv"
os.makedirs(CSV_FOLDER, exist_ok=True)


os.makedirs("log", exist_ok=True) # Create log folder if doesn't exit
# dynamically names a log file using the current date & time
LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

# method to add log msg into log file & prints them to console
def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f: # "a" mode → appends messages instead of overwriting
        f.write(msg + "\n")
    print(msg)

### Create DuckDB Connection

In [26]:
import os
import duckdb

DB_FOLDER = "database"
DB_PATH = os.path.join(DB_FOLDER, "cse_data.db")

con = duckdb.connect(database=DB_PATH)

In [27]:
# ================ Creating labels for 10-day return ==========================================================

# Create labeled table with:
#  - future closing price (10 days ahead)
#  - future return %
#  - classification label (buy or not)
#
# f""" allows Python to insert variables like {PROFIT_THRESHOLD} into SQL
#
# Using a CTE (temp) to compute future price once for efficiency
# instead of recalculating LEAD() multiple times
#
# LEAD(close, 10)
#     → gets closing price 10 rows ahead / after the current date (actualy not not days, rcords) if today 26.03.05 then look for 26.03.15
# PARTITION BY symbol
#     → calculation done separately for each stock
# ORDER BY date
#     → ensures rows are in time order
#
# Return formula:
#     future_return = (future_price / current_price) - 1
#
# Target label rule:
#     return ≥ threshold → 1 (profitable → buy signal)
#     return < threshold → 0 (not profitable)

log("=== Phase 3: Creating labels for 10-day return ===")


cols = ["target_buy_10d"]
for c in cols:
    con.execute(f"ALTER TABLE stocks_features_table DROP COLUMN IF EXISTS {c}")
con.execute("DROP TABLE IF EXISTS temp_future_price_table") 
con.execute("DROP TABLE IF EXISTS temp_returns_table") 
con.execute("DROP TABLE IF EXISTS stocks_features_labeled_10d_table") 


# Threshold for profitable trade, 0.05 - 5% profit in 10 days
PROFIT_THRESHOLD = 0.05

# Computing future closing price (10 records ahead/future),we cant here use past data bcs we wanna train model for future
# Right we dont have price record for lat 10 days if today 20/03 no data from 11-today(20) / 03 but that is ok we use that for training the model no 
# need to all at advanced we can use that trained model for today prediction although it is old 10 day, we train model from 1st day to today - 10days, 
# totaly fine
con.execute("""
    CREATE OR REPLACE TABLE temp_future_price_table AS
    SELECT 
        symbol,
        date, 
        close,
            -- looks n rows ahead (future)
            LEAD(close, 10) OVER (PARTITION BY symbol ORDER BY date) AS future_close_10d
    FROM stocks_features_clean_table
    WHERE close > 0
""")

# Compute future return
# After 10 days how the actual return has happeneded, is it a gain or loss relative to the today price if today price 100, after day days 
# it has become 110, then return is (110/100) - 1 = 1.1 - 1 = 0.1 is the gain
con.execute(f"""
    CREATE OR REPLACE TABLE temp_returns_table AS
    SELECT 
        symbol,
        date,
        future_close_10d,
        close,
           ((future_close_10d / close) - 1) AS raw_future_return,

        CASE 
           WHEN (future_close_10d / close - 1) > 1 THEN 1
           WHEN (future_close_10d / close - 1) < -1 THEN -1
           ELSE (future_close_10d / close - 1)
        END AS future_return_10d,

        -- if calculated future_return_10d value is > PROFIT_THRESHOLD value >>> 1 can buy
        COALESCE(
            CASE 
            WHEN (future_close_10d / close - 1) >= {PROFIT_THRESHOLD} THEN 1
            ELSE 0
            END, 0
        ) AS target_buy_10d
       
    FROM temp_future_price_table
    WHERE future_close_10d IS NOT NULL
""")

# copy stocks_features_clean_table to new table named stocks_features_labeled_10d_table to add target features
con.execute(f"""
    CREATE OR REPLACE TABLE stocks_features_labeled_10d_table AS
    SELECT *,
    FROM stocks_features_clean_table;
""")

# Crate support columns
con.execute("ALTER TABLE stocks_features_labeled_10d_table ADD COLUMN IF NOT EXISTS future_close_10d DOUBLE")
con.execute("ALTER TABLE stocks_features_labeled_10d_table ADD COLUMN IF NOT EXISTS future_return_10d DOUBLE")

# Create label columns
con.execute("ALTER TABLE stocks_features_labeled_10d_table ADD COLUMN IF NOT EXISTS target_buy_10d INTEGER")

# target_buy_10d - target/y column, buy or not 1 or 0
con.execute("""
    UPDATE stocks_features_labeled_10d_table AS s
    SET 
        -- COALESCE(value1, value2, ..., valueN) returns the first non-NULL value from the list
        target_buy_10d = COALESCE(v.target_buy_10d, 0),
        future_close_10d = v.future_close_10d,
        future_return_10d = v.future_return_10d
    FROM temp_returns_table AS v
    WHERE s.symbol = v.symbol AND s.date = v.date;
""")

con.execute("""
    UPDATE stocks_features_labeled_10d_table
    SET target_buy_10d = 0
    WHERE target_buy_10d IS NULL;
""")

log("Label creation complete : 10-day later closed price col, return % col & target_buy_10d binary col added")


# ================= EXPORT LABELED DATASET =====================================================================

CSV_FOLDER = "output_csv"
os.makedirs(CSV_FOLDER, exist_ok=True)
LABELED_CSV_PATH = os.path.join(CSV_FOLDER, "features_labeled_10d.csv")

log(f"Exporting labeled dataset to {LABELED_CSV_PATH} ...")

# HEADER → includes column names
# DELIMITER ',' → comma-separated format
con.execute(f"""
    COPY stocks_features_labeled_10d_table
    TO '{LABELED_CSV_PATH}'
    (HEADER, DELIMITER ',');
""")

log(f"Labeled dataset exported dataset ready for ML training → {LABELED_CSV_PATH}")



# ================= QUICK PREVIEW =====================================================================================

# temp_future_price_df = con.execute("SELECT * FROM temp_future_price_table").fetchdf()
# print("\n============= temp_future_price_table =============\n", temp_future_price_df.head(25))

temp_future_price_df = con.execute("SELECT * FROM temp_returns_table").fetchdf()
print("\n============= temp_returns_table =============\n", temp_future_price_df.sample(25))

stocks_features_labeled_df = con.execute("SELECT * FROM stocks_features_labeled_10d_table").fetchdf()
print("\n============= stocks_features_labeled_10d_table =============\n", stocks_features_labeled_df.sample(25))

print("\no of columns : ", len(stocks_features_labeled_df.columns),"\n",stocks_features_labeled_df.columns)

# table > temp_future_price   temp_returns
# col   > future_return_10d

=== Phase 3: Creating labels for 10-day return ===
Label creation complete : 10-day later closed price col, return % col & target_buy_10d binary col added
Exporting labeled dataset to output_csv\features_labeled_10d.csv ...
Labeled dataset exported dataset ready for ML training → output_csv\features_labeled_10d.csv

============= temp_returns_table =============
           symbol       date  future_close_10d    close  raw_future_return  \
5087  ONAL.N0000 2026-02-25             37.90    42.00          -0.097619   
7909  CITW.N0000 2026-01-29              2.00     2.00           0.000000   
5629  SFCL.N0000 2026-02-19            504.50   826.75          -0.389779   
1177  BREW.N0000 2026-01-23           2750.50  2646.00           0.039494   
6656  LITE.N0000 2026-02-16             48.70    53.00          -0.081132   
8123  RENU.N0000 2026-02-09            915.25   957.25          -0.043876   
1381  INME.N0000 2026-02-24           1415.25  1826.25          -0.225051   
7309  CTEA.N0000 2

### Class Imbalance Testing

In [28]:
stocks_features_labeled_df.tail(10)

,company,symbol,volume,trades,prev_close,open,high,low,close,change,change_percentage,date,dup_index,close_1d,close_3d,close_5d,close_10d,close_20d,close_60d,high_20,low_20,std_vol_20,ret_1,ret_3,ret_5,ret_10,ret_20,ret_60,std_close_5,std_close_10,std_close_20,range_5,atr_14,TR,volume_ratio,volume_z,avg_vol_10,avg_vol_20,liquidity_score,ma_ratio_5,ma_ratio_10,ma_ratio_20,price_position,breakout_flag,momentum_score,volatility_score,turnover_ratio,trend_angle,range_position,liquidity_rank,future_close_10d,future_return_10d,target_buy_10d
11776,VIDULLANKA PLC,VLL.X0000,6651,19,17.9,18.0,18.0,17.8,17.8,-0.1,-0.56,2026-02-27,1,17.9,18.1,18.0,18.3,18.5,NaN,18.9,17.8,24654.721935,-0.005587,-0.016575,-0.011111,-0.027322,-0.037838,NaN,0.158114,0.200000,0.346258,0.5,0.364286,0.2,0.470871,-0.303143,10796.4,14124.90,0.142741,0.988889,0.983425,0.978560,0.240000,0,-0.011591,0.208309,0.002857,-0.006466,0.000000,212.0,NaN,NaN,0
11777,VIDULLANKA PLC,VLL.X0000,953,6,18.2,18.3,18.3,17.9,17.9,-0.3,-1.65,2026-02-28,1,17.8,18.2,18.0,18.2,17.8,NaN,18.9,17.8,12482.441841,0.005618,-0.016484,-0.005556,-0.016484,0.005618,NaN,0.164317,0.205751,0.341012,0.5,0.357143,0.5,0.106692,-0.639238,10754.3,8932.25,0.068202,0.995551,0.990592,0.983787,0.388889,0,-0.002154,0.212086,0.006296,-0.016767,0.090909,223.0,NaN,NaN,0
11778,VIDULLANKA PLC,VLL.X0000,123595,22,17.8,17.9,18.4,17.7,17.8,0.0,0.00,2026-03-04,1,17.9,17.9,18.1,18.4,17.9,NaN,18.9,17.8,28453.690366,-0.005587,-0.005587,-0.016575,-0.032609,-0.005587,NaN,0.164317,0.185293,0.346258,0.7,0.385714,0.7,8.306953,3.820822,22048.4,14878.50,31.739393,0.993304,0.988340,0.978560,0.076923,0,-0.014287,0.206998,0.000178,-0.027068,0.000000,211.0,NaN,NaN,0
11779,VIDULLANKA PLC,VLL.X0000,13077,27,17.8,17.8,17.8,17.2,17.4,-0.4,-2.25,2026-03-05,1,17.8,17.8,18.2,18.4,17.8,NaN,18.9,17.4,28448.876725,-0.022472,-0.022472,-0.043956,-0.054348,-0.022472,NaN,0.207364,0.218327,0.379889,1.2,0.392857,0.6,0.875697,-0.065249,17725.0,14933.25,0.057138,0.979730,0.971524,0.957622,-0.230769,0,-0.035292,0.245158,0.002065,-0.044511,0.000000,212.0,NaN,NaN,0
11780,VIDULLANKA PLC,VLL.X0000,476,4,17.0,17.2,17.3,17.2,17.3,0.3,1.76,2026-03-06,1,17.4,17.9,17.9,18.0,17.8,NaN,18.9,17.3,28596.484264,-0.005747,-0.033520,-0.033520,-0.038889,-0.028090,NaN,0.270185,0.287518,0.419868,1.2,0.392857,0.2,0.034147,-0.470820,16330.1,13939.80,0.016077,0.980726,0.969731,0.953431,0.055556,0,-0.020707,0.305322,0.008403,-0.062782,0.000000,214.0,NaN,NaN,0
11781,VIDULLANKA PLC,VLL.X0000,56783,74,17.3,17.3,17.3,16.0,16.5,-0.8,-4.62,2026-03-09,1,17.3,17.8,17.8,18.0,18.3,NaN,18.9,16.5,29943.196724,-0.046243,-0.073034,-0.073034,-0.083333,-0.098361,NaN,0.554076,0.504315,0.555807,2.4,0.457143,1.3,3.389655,1.336903,21993.0,16751.85,4.531641,0.949367,0.932730,0.913874,-0.388889,0,-0.061698,0.539494,0.001303,-0.083835,0.000000,207.0,NaN,NaN,0
11782,VIDULLANKA PLC,VLL.X0000,12061,23,16.5,16.9,18.0,16.9,17.4,0.9,5.45,2026-03-10,1,16.5,17.4,17.9,18.0,18.9,NaN,18.9,16.5,29833.128650,0.054545,0.000000,-0.027933,-0.033333,-0.079365,NaN,0.476445,0.498999,0.536656,2.4,0.521429,1.5,0.704580,-0.169510,22248.6,17118.00,0.119433,1.006944,0.986954,0.967742,0.466667,0,0.012226,0.495254,0.001907,-0.079850,0.375000,207.0,NaN,NaN,0
11783,VIDULLANKA PLC,VLL.X0000,9774,19,17.4,17.6,17.6,16.5,16.5,-0.9,-5.17,2026-03-11,1,17.4,17.3,17.8,18.1,18.6,NaN,18.9,16.5,29875.625760,-0.051724,-0.046243,-0.073034,-0.088398,-0.112903,NaN,0.476445,0.581282,0.609465,2.0,0.564286,1.1,0.579889,-0.237014,22998.0,16854.95,0.137442,0.969448,0.944476,0.923077,0.166667,0,-0.065452,0.534500,0.001944,-0.091203,0.000000,207.0,NaN,NaN,0
11784,VIDULLANKA PLC,VLL.X0000,30359,64,16.7,16.4,16.5,15.0,15.9,-0.8,-4.79,2026-03-13,1,16.5,16.5,17.4,18.2,18.9,NaN,18.5,15.9,29806.552350,-0.036364,-0.036364,-0.086207,-0.126374,-0.158730,NaN,0.626099,0.702693,0.705523,3.0,0.635714,1.5,1.661449,0.405495,25468.2,18272.60,0.673709,0.950957,0.922274,0.897038,-0.034483,0,-0.069319,0.664962,0.002108,-0.103233,0.000000,203.0,NaN,NaN,0
11785,VIDUL

In [29]:
import duckdb

# Load db
con = duckdb.connect("database/cse_data.db")

counts = con.execute("""
    SELECT 
        target_buy_10d,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM stocks_features_labeled_10d_table
    GROUP BY target_buy_10d
    ORDER BY target_buy_10d
""").fetch_df()

print(counts)

   target_buy_10d  count  percentage
0               0  10296       87.36
1               1   1490       12.64


### Close DuckDB Connection

In [30]:
if 'con' in globals():  # Check if connection exists
    try:
        con.close()        # Close it
        print("DuckDB connection closed.")
    except Exception as e:
        print("Error closing DuckDB:", e)
    finally:
        del con             # Delete the variable from memory

# stocks_features_labeled_10d_table

DuckDB connection closed.
